In [ ]:
# Customer Support Ticket Auto-Triage System

In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import time
import pickle
import json

2025-09-27 16:58:31.703336: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-27 16:58:31.703668: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-27 16:58:31.751579: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-27 16:58:33.418119: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [2]:
np.random.seed(42)

n_samples = 5000
categories = ['Bug Report', 'Feature Request', 'Technical Issue', 'Billing Inquiry', 'Account Management']
priorities = ['Low', 'Medium', 'High', 'Critical']

# Sample templates for each category
templates = {
    'Bug Report': [
        ('App crashes when uploading', 'The application crashes every time I try to upload a file larger than 10MB'),
        ('Error 404 on dashboard', 'Getting error 404 when accessing the main dashboard page'),
        ('Login button not working', 'Cannot click the login button on mobile devices')
    ],
    'Feature Request': [
        ('Add dark mode', 'Would love to have a dark mode option for night time usage'),
        ('Export to PDF needed', 'Please add functionality to export reports as PDF'),
        ('Bulk upload feature', 'Need ability to upload multiple files at once')
    ],
    'Technical Issue': [
        ('API connection timeout', 'Getting timeout errors when connecting to the API endpoint'),
        ('Database sync issues', 'Data not syncing properly between mobile and web versions'),
        ('Performance degradation', 'System becomes very slow after 2 hours of usage')
    ],
    'Billing Inquiry': [
        ('Wrong charge amount', 'I was charged $99 instead of the advertised $79'),
        ('Refund request', 'Need a refund for accidental duplicate subscription'),
        ('Payment method update', 'Unable to update my credit card information')
    ],
    'Account Management': [
        ('Password reset not working', 'Not receiving password reset emails'),
        ('Delete account request', 'Please help me delete my account permanently'),
        ('Change email address', 'Need to update my registered email address')
    ]
}

data = []
for i in range(n_samples):
    category = np.random.choice(categories)
    subject, description = templates[category][np.random.randint(0, 3)]
    
    # Add variations
    subject += f" #{i%100}"
    description += f" Issue ID: {i}. This has been happening for {np.random.randint(1,30)} days."
    
    data.append({
        'Ticket_ID': i + 1000,
        'Subject': subject,
        'Description': description,
        'Category': category,
        'Priority': np.random.choice(priorities),
        'Timestamp': pd.Timestamp.now() - pd.Timedelta(days=np.random.randint(0, 365))
    })

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:\n{df['Category'].value_counts()}")

Dataset shape: (5000, 6)

Category distribution:
Category
Feature Request       1051
Account Management    1004
Technical Issue        989
Billing Inquiry        978
Bug Report             978
Name: count, dtype: int64


In [3]:
df['text'] = df['Subject'] + ' ' + df['Description']

label_encoder = LabelEncoder()
df['category_encoded'] = label_encoder.fit_transform(df['Category'])

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, 
    df['category_encoded'].values,
    test_size=0.2, 
    random_state=42,
    stratify=df['category_encoded']
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 4000
Test samples: 1000


In [4]:
max_features = 5000
sequence_length = 200

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

vectorize_layer.adapt(X_train)

vocab = vectorize_layer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 4146


2025-09-27 16:59:23.478306: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
